# Lezione 7.1 - Riconoscimento di Cifre Scritte a Mano con MNIST 🤖

In questa lezione costruiremo una **Rete Neurale Convoluzionale (CNN)** per riconoscere cifre scritte a mano (da 0 a 9) usando il famoso dataset MNIST.

## Cosa faremo:

1. 📥 Importare e esplorare il dataset MNIST
2. 👀 Visualizzare alcune immagini con Matplotlib
3. 🔧 Preparare i dati per l'addestramento
4. 🧠 Costruire una CNN semplice con PyTorch
5. 🏋️ Addestrare la rete neurale
6. 📊 Valutare i risultati

## Cos'è MNIST?

MNIST (Modified National Institute of Standards and Technology) è un dataset di 70.000 immagini di cifre scritte a mano:
- **60.000 immagini** per l'addestramento
- **10.000 immagini** per il test
- Ogni immagine è **28×28 pixel** in scala di grigi
- Le cifre vanno da **0 a 9** (10 classi)

---

## 1️⃣ Importazione delle Librerie

Iniziamo importando tutte le librerie necessarie:

In [ ]:
# Librerie per la gestione dati
import pandas as pd
import numpy as np

# Librerie per la visualizzazione
import matplotlib.pyplot as plt

# Librerie per PyTorch (deep learning)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

print("✅ Librerie importate con successo!")
print(f"Versione PyTorch: {torch.__version__}")

---

## 2️⃣ Download del Dataset MNIST da Kaggle

Il dataset MNIST è disponibile su Kaggle. Scaricheremo la versione CSV che è più comune quando si lavora con dataset reali.

### Setup dell'API di Kaggle

Per scaricare dataset da Kaggle, hai bisogno di:
1. Un account Kaggle (gratuito)
2. Il file `kaggle.json` con le tue credenziali API

**Come ottenere kaggle.json:**
1. Vai su [kaggle.com](https://www.kaggle.com)
2. Clicca sulla tua icona profilo → Account → API → "Create New API Token"
3. Scaricherai il file `kaggle.json`
4. Posizionalo in `~/.kaggle/kaggle.json` (Linux/Mac) o `C:\Users\<username>\.kaggle\kaggle.json` (Windows)

In [ ]:
# Installiamo la libreria Kaggle se non è già installata
import subprocess
import sys

try:
    import kaggle
    print("✅ Kaggle API già installata")
except ImportError:
    print("📦 Installazione Kaggle API...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle", "-q"])
    import kaggle
    print("✅ Kaggle API installata!")

# Verifichiamo che le credenziali siano configurate
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    print("✅ Autenticazione Kaggle riuscita!")
except Exception as e:
    print("❌ Errore nell'autenticazione Kaggle:")
    print(f"   {e}")
    print("\n💡 Assicurati di aver:")
    print("   1. Creato un account su kaggle.com")
    print("   2. Scaricato il file kaggle.json dalle impostazioni API")
    print("   3. Posizionato kaggle.json in ~/.kaggle/ (Linux/Mac) o C:\\Users\\<username>\\.kaggle\\ (Windows)")

### Download del Dataset

Ora scarichiamo il dataset MNIST in formato CSV da Kaggle:

---

## 3️⃣ Caricamento dei Dati con Pandas

Ora che abbiamo scaricato i file CSV, carichiamoli con Pandas. Questo è il punto di partenza quando lavori con dataset reali!

In [ ]:
# Carichiamo i CSV con Pandas
print("📊 Caricamento dei dataset CSV...")

# I file scaricati da Kaggle si chiamano mnist_train.csv e mnist_test.csv
train_df = pd.read_csv('data/mnist_train.csv')
test_df = pd.read_csv('data/mnist_test.csv')

print("✅ Dataset caricati con successo!")
print(f"\n📦 Training set: {train_df.shape}")
print(f"   - {train_df.shape[0]:,} immagini")
print(f"   - {train_df.shape[1]} colonne (1 label + 784 pixel)")

print(f"\n📦 Test set: {test_df.shape}")
print(f"   - {test_df.shape[0]:,} immagini")
print(f"   - {test_df.shape[1]} colonne")

print(f"\n? Prime righe del training set:")
train_df.head()

### ? Struttura del Dataset

Il CSV di Kaggle ha questa struttura:
- **Colonna 0**: `label` (la cifra da 0 a 9)
- **Colonne 1-784**: `pixel0`, `pixel1`, ..., `pixel783` (i valori dei pixel dell'immagine 28×28 appiattita)
- **Valori pixel**: da 0 (nero) a 255 (bianco)

In [ ]:
# Verifichiamo i nomi delle colonne
print("📋 Colonne del dataset:")
print(f"   Prima colonna: '{train_df.columns[0]}' (dovrebbe essere 'label')")
print(f"   Totale colonne pixel: {len(train_df.columns) - 1}")
print(f"   Range colonne: {train_df.columns[1]} ... {train_df.columns[-1]}")

# Verifichiamo il formato
label_col = train_df.columns[0]
pixel_cols = train_df.columns[1:]

print(f"\n✅ Formato riconosciuto correttamente!")

### Analisi Esplorativa dei Dati (EDA)

Vediamo la distribuzione delle classi e le statistiche di base:

---

## 4️⃣ Visualizzazione delle Immagini

Visualizziamo alcune immagini dal DataFrame per capire meglio i dati:

In [ ]:
# Funzione per visualizzare un'immagine dal DataFrame
def show_image_from_row(row_data, label_col):
    """Converte una riga del DataFrame in immagine 28x28"""
    label = row_data[label_col]
    # Prendiamo tutti i pixel (tutte le colonne tranne la label)
    pixels = row_data.drop(label_col).values
    img = pixels.reshape(28, 28)
    return img, label

# Visualizziamo 15 immagini casuali
fig, axes = plt.subplots(3, 5, figsize=(12, 7))

for i, ax in enumerate(axes.flat):
    # Prendiamo una riga casuale
    idx = np.random.randint(0, len(train_df))
    img, label = show_image_from_row(train_df.iloc[idx], label_col)
    
    # Visualizziamo l'immagine
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Cifra: {label}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.suptitle('Esempi di Cifre dal Dataset Kaggle', fontsize=16, y=1.02)
plt.show()

### Analisi Dettagliata di un'Immagine

In [ ]:
# Prendiamo la prima immagine per analizzarla in dettaglio
img, label = show_image_from_row(train_df.iloc[0], label_col)

print(f"📸 Forma dell'immagine: {img.shape}")
print(f"🏷️  Etichetta (cifra): {label}")
print(f"📊 Range valori pixel: [{img.min()}, {img.max()}]")
print(f"📈 Media pixel: {img.mean():.2f}")
print(f"📉 Deviazione standard: {img.std():.2f}")

# Visualizziamo l'immagine e la heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Immagine
ax1.imshow(img, cmap='gray')
ax1.set_title(f'Immagine della cifra: {label}', fontsize=14)
ax1.axis('off')

# Heatmap dei valori pixel
im = ax2.imshow(img, cmap='viridis')
ax2.set_title('Valori dei pixel (0=nero, 255=bianco)', fontsize=14)
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.show()

---

## 5️⃣ Creazione di un Custom Dataset per PyTorch

Ora viene la parte importante! Creiamo una **classe Custom Dataset** che PyTorch può usare. Questo è essenziale quando lavori con i tuoi dati.

### Cos'è un Custom Dataset?
È una classe Python che dice a PyTorch:
- Come caricare un singolo dato (immagine + label)
- Quanti dati ci sono in totale
- Come preprocessare i dati

In [ ]:
class MNISTDataset(Dataset):
    """
    Custom Dataset per caricare MNIST da CSV.
    
    Questo approccio funziona con QUALSIASI dataset in formato CSV!
    Basta modificare la classe per adattarla ai tuoi dati.
    """
    
    def __init__(self, csv_file, label_col='label'):
        """
        Args:
            csv_file (string): Path al file CSV
            label_col (string): Nome della colonna con le label
        """
        print(f"📂 Caricamento dataset da {csv_file}...")
        
        # Leggiamo il CSV con Pandas
        self.data = pd.read_csv(csv_file)
        
        # Separiamo le label dai pixel
        self.label_col = label_col
        self.labels = self.data[label_col].values
        self.pixels = self.data.drop(label_col, axis=1).values
        
        print(f"   ✅ Caricati {len(self.data)} campioni")
        
    def __len__(self):
        """Ritorna il numero totale di campioni"""
        return len(self.data)
    
    def __getitem__(self, idx):
        """
        Ritorna un singolo campione (immagine, label).
        Questo metodo viene chiamato ogni volta che carichi un dato!
        
        Args:
            idx (int): Indice del campione
            
        Returns:
            image (Tensor): Immagine come tensore [1, 28, 28]
            label (int): Etichetta della cifra
        """
        # Prendiamo i pixel e li reshaping in 28x28
        pixels = self.pixels[idx].reshape(28, 28)
        
        # PREPROCESSING: normalizziamo i valori da [0, 255] a [0, 1]
        # Questo aiuta la rete neurale ad addestrare meglio!
        pixels = pixels.astype(np.float32) / 255.0
        
        # Convertiamo in tensore PyTorch e aggiungiamo la dimensione del canale
        # Da (28, 28) a (1, 28, 28) perché le CNN si aspettano [Canali, Altezza, Larghezza]
        image = torch.from_numpy(pixels).unsqueeze(0)
        
        # Prendiamo la label
        label = int(self.labels[idx])
        
        return image, label


# Creiamo i dataset custom dai file CSV di Kaggle
print("🔨 Creazione dei Custom Dataset...\n")

train_dataset = MNISTDataset('data/mnist_train.csv', label_col=label_col)
test_dataset = MNISTDataset('data/mnist_test.csv', label_col=label_col)

print(f"\n✅ Custom Dataset pronti!")

# Testiamo il dataset
img, label = train_dataset[0]
print(f"\n🔍 Test del dataset:")
print(f"   Shape immagine: {img.shape}")
print(f"   Tipo: {img.dtype}")
print(f"   Range valori: [{img.min():.3f}, {img.max():.3f}]")
print(f"   Label: {label}")

### Creazione dei DataLoader

Ora creiamo i DataLoader che useranno il nostro Custom Dataset:

---

## 6️⃣ Costruzione della Rete Neurale Convoluzionale (CNN)

Ora costruiamo la nostra CNN! Sarà molto semplice ma efficace.

### Architettura della rete:

1. **Conv Layer 1**: Trova 16 pattern nelle immagini (es. bordi, curve)
2. **Pooling 1**: Riduce le dimensioni (downsampling)
3. **Conv Layer 2**: Trova 32 pattern più complessi
4. **Pooling 2**: Riduce ancora le dimensioni
5. **Fully Connected**: Combina i pattern per fare la predizione finale (0-9)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Layer convoluzionali
        # Conv1: da 1 canale (grigio) a 16 filtri, kernel 3x3
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        # Conv2: da 16 a 32 filtri
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        # Pooling: riduce le dimensioni di 2x
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        # Dopo 2 pooling: 28 -> 14 -> 7, quindi 7x7x32 = 1568 neuroni
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)  # 10 classi (0-9)
        
        # Funzione di attivazione
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Conv1 + ReLU + Pool
        x = self.pool(self.relu(self.conv1(x)))  # 28x28 -> 14x14
        # Conv2 + ReLU + Pool
        x = self.pool(self.relu(self.conv2(x)))  # 14x14 -> 7x7
        
        # Appiattimento: da (batch, 32, 7, 7) a (batch, 1568)
        x = x.view(-1, 32 * 7 * 7)
        
        # Fully connected layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Output: 10 neuroni (uno per classe)
        
        return x

# Creiamo il modello
model = SimpleCNN()
print("✅ Modello creato!")
print(f"\n📋 Architettura della rete:\n")
print(model)

### Contiamo i parametri del modello

In [ ]:
# Contiamo quanti parametri (pesi) ha la nostra rete
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🔢 Parametri totali: {total_params:,}")
print(f"🎯 Parametri addestrabili: {trainable_params:,}")

---

## 7️⃣ Configurazione dell'Addestramento

Prima di addestrare, dobbiamo definire:

1. **Loss Function** (Funzione di perdita): misura quanto sbaglia la rete
2. **Optimizer** (Ottimizzatore): aggiorna i pesi per ridurre la loss
3. **Device**: CPU o GPU (se disponibile)

In [ ]:
# 1. Verifichiamo se abbiamo una GPU disponibile
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device utilizzato: {device}")

# Spostiamo il modello sul device
model = model.to(device)

# 2. Loss function: CrossEntropyLoss per classificazione multi-classe
criterion = nn.CrossEntropyLoss()

# 3. Optimizer: Adam con learning rate 0.001
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✅ Configurazione completata!")
print(f"📉 Loss function: CrossEntropyLoss")
print(f"⚙️  Optimizer: Adam (lr=0.001)")

---

## 8️⃣ Addestramento del Modello

Ora alleniamo la rete! Faremo 5 epoche (5 passaggi completi sul dataset).

### Cosa succede durante l'addestramento:
1. **Forward pass**: la rete fa una predizione
2. **Calcolo della loss**: misuriamo l'errore
3. **Backward pass**: calcoliamo i gradienti
4. **Update**: aggiorniamo i pesi

In [ ]:
num_epochs = 5
train_losses = []
train_accuracies = []

print("🏋️ Inizio addestramento...\n")

for epoch in range(num_epochs):
    model.train()  # Modalità training
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        # Spostiamo i dati sul device (GPU o CPU)
        images, labels = images.to(device), labels.to(device)
        
        # 1. Azzeriamo i gradienti
        optimizer.zero_grad()
        
        # 2. Forward pass
        outputs = model(images)
        
        # 3. Calcoliamo la loss
        loss = criterion(outputs, labels)
        
        # 4. Backward pass
        loss.backward()
        
        # 5. Update dei pesi
        optimizer.step()
        
        # Statistiche
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Stampiamo il progresso ogni 100 batch
        if (batch_idx + 1) % 100 == 0:
            print(f'Epoca [{epoch+1}/{num_epochs}], '
                  f'Batch [{batch_idx+1}/{len(train_loader)}], '
                  f'Loss: {loss.item():.4f}')
    
    # Statistiche per epoca
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    print(f'\n✅ Epoca {epoch+1} completata!')
    print(f'   Loss media: {epoch_loss:.4f}')
    print(f'   Accuracy: {epoch_acc:.2f}%\n')
    print('-' * 60)

print("\n🎉 Addestramento completato!")

### Visualizziamo l'andamento dell'addestramento

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grafico della Loss
ax1.plot(range(1, num_epochs + 1), train_losses, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Epoca', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss durante l\'addestramento', fontsize=14)
ax1.grid(True, alpha=0.3)

# Grafico dell'Accuracy
ax2.plot(range(1, num_epochs + 1), train_accuracies, 'g-o', linewidth=2, markersize=8)
ax2.set_xlabel('Epoca', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Accuracy durante l\'addestramento', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📈 Loss finale: {train_losses[-1]:.4f}")
print(f"🎯 Accuracy finale sul training set: {train_accuracies[-1]:.2f}%")

---

## 9️⃣ Valutazione sul Test Set

Ora testiamo il modello su dati **mai visti prima** (il test set) per vedere quanto generalizza bene.

In [ ]:
model.eval()  # Modalità valutazione (disattiva dropout, batch norm, ecc.)

correct = 0
total = 0

# torch.no_grad() disattiva il calcolo dei gradienti (non serve durante il test)
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        
        # Prendiamo la classe con probabilità più alta
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"🎯 Accuracy sul Test Set: {test_accuracy:.2f}%")
print(f"✅ Predizioni corrette: {correct}/{total}")

---

## 🔟 Visualizzazione dei Risultati

Vediamo alcune predizioni della rete, sia corrette che sbagliate!

In [ ]:
# Prendiamo un batch dal test set
dataiter = iter(test_loader)
images, labels = next(dataiter)

# Facciamo le predizioni
images = images.to(device)
outputs = model(images)
_, predicted = torch.max(outputs, 1)

# Visualizziamo 15 esempi
fig, axes = plt.subplots(3, 5, figsize=(14, 8))

for i, ax in enumerate(axes.flat):
    # Convertiamo l'immagine in numpy
    img = images[i].cpu().squeeze().numpy()
    true_label = labels[i].item()
    pred_label = predicted[i].cpu().item()
    
    # Visualizziamo l'immagine
    ax.imshow(img, cmap='gray')
    
    # Titolo verde se corretto, rosso se sbagliato
    if true_label == pred_label:
        color = 'green'
        title = f'✓ Predetto: {pred_label}'
    else:
        color = 'red'
        title = f'✗ Pred: {pred_label} (Vero: {true_label})'
    
    ax.set_title(title, color=color, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.suptitle('Esempi di Predizioni (Verde=Corretto, Rosso=Sbagliato)', 
             fontsize=16, y=1.02)
plt.show()

### Matrice di Confusione

La matrice di confusione ci mostra quali cifre vengono confuse più spesso tra loro.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Raccogliamo tutte le predizioni e le etichette vere
all_predictions = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Creiamo la matrice di confusione
cm = confusion_matrix(all_labels, all_predictions)

# Visualizziamo
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.title('Matrice di Confusione', fontsize=16)
plt.ylabel('Etichetta Vera', fontsize=12)
plt.xlabel('Etichetta Predetta', fontsize=12)
plt.show()

# Accuracy per classe
print("\n📊 Accuracy per singola cifra:")
for i in range(10):
    class_correct = cm[i, i]
    class_total = cm[i, :].sum()
    class_acc = 100 * class_correct / class_total
    print(f"Cifra {i}: {class_acc:.2f}% ({class_correct}/{class_total})")

---

## 🎓 Riepilogo e Conclusioni

### Cosa abbiamo fatto:

1. ✅ **Scaricato** e **convertito** il dataset MNIST in formato CSV
2. ✅ **Caricato** i dati con Pandas (come faresti con qualsiasi dataset da Kaggle)
3. ✅ **Esplorato** i dati con statistiche e visualizzazioni
4. ✅ **Creato un Custom Dataset** per PyTorch (fondamentale!)
5. ✅ **Costruito** una CNN semplice ma efficace
6. ✅ **Addestrato** la rete per 5 epoche
7. ✅ **Valutato** le performance sul test set
8. ✅ **Visualizzato** i risultati con grafici e matrice di confusione

### Risultati:

- **Test Accuracy**: ~98-99% (eccellente per un modello così semplice!)
- **Parametri**: ~200.000 (molto leggero)
- **Tempo di addestramento**: pochi minuti su CPU

### 🔑 Concetti chiave appresi:

- **Pandas per caricare CSV**: il modo più comune di gestire dataset
- **Custom Dataset**: come creare la tua classe Dataset per PyTorch
- **Preprocessing**: normalizzazione e conversione in tensori
- **DataLoader**: caricamento efficiente dei dati in batch
- **CNN**: rete specializzata per immagini
- **Training loop**: forward → loss → backward → update
- **Valutazione**: sempre testare su dati mai visti prima

### 📝 Workflow completo per i tuoi dataset:

1. **Scarica** i dati (Kaggle, siti web, ecc.)
2. **Carica** con Pandas (`pd.read_csv()`)
3. **Esplora** i dati (distribuzioni, valori mancanti, ecc.)
4. **Crea un Custom Dataset** (ereditando da `torch.utils.data.Dataset`)
5. **Crea i DataLoader**
6. **Addestra** il modello
7. **Valuta** i risultati

### Possibili miglioramenti:

1. 📈 Aumentare il numero di epoche (10-15)
2. 🔧 Provare diversi learning rate
3. 🧠 Aggiungere più layer convoluzionali
4. 🎯 Usare data augmentation (rotazioni, traslazioni)
5. 💾 Salvare il modello addestrato
6. 📊 Implementare early stopping

### Prossimi passi:

- Provare con **i tuoi dataset** (immagini, CSV, ecc.)
- Implementare **data augmentation** avanzata
- Usare **transfer learning** con modelli pre-addestrati
- Gestire **dataset sbilanciati** (weighted loss)

---

## 💾 Bonus: Salvataggio del Modello

Salviamo il modello addestrato per poterlo riutilizzare in futuro senza doverlo riaddestrare.

In [ ]:
# Salviamo il modello
torch.save(model.state_dict(), 'mnist_cnn_model.pth')
print("✅ Modello salvato in 'mnist_cnn_model.pth'")

# Per ricaricare il modello in futuro:
# model = SimpleCNN()
# model.load_state_dict(torch.load('mnist_cnn_model.pth'))
# model.eval()
# print("✅ Modello caricato!")

---

## 🎯 Esercizi Suggeriti

1. **Modifica il learning rate**: prova con `lr=0.01` o `lr=0.0001` e osserva le differenze
2. **Aumenta le epoche**: allena per 10 epoche e confronta i risultati
3. **Aggiungi un layer**: inserisci un terzo layer convoluzionale
4. **Data augmentation**: aggiungi rotazioni casuali alle immagini durante il training
5. **Prova Fashion-MNIST**: sostituisci MNIST con Fashion-MNIST (vestiti invece di cifre)
6. **Analizza gli errori**: trova le 10 immagini dove il modello sbaglia di più e analizzale

---

### 📚 Riferimenti

- [PyTorch Documentation](https://pytorch.org/docs/)
- [MNIST Database](http://yann.lecun.com/exdb/mnist/)
- [Convolutional Neural Networks Explained](https://cs231n.github.io/)

---

**Fine della Lezione 7.1** 🎉

In [ ]:
# Analisi Esplorativa dei Dati (EDA)

print("📊 Distribuzione delle cifre nel training set:")
print(train_df[label_col].value_counts().sort_index())

print(f"\n📊 Distribuzione delle cifre nel test set:")
print(test_df[label_col].value_counts().sort_index())

# Statistiche sui pixel
print(f"\n📈 Statistiche sui valori dei pixel (training set):")
print(f"   Range: [{train_df[pixel_cols].min().min()}, {train_df[pixel_cols].max().max()}]")
print(f"   Media generale: {train_df[pixel_cols].mean().mean():.2f}")
print(f"   Std generale: {train_df[pixel_cols].std().mean():.2f}")

# Verifichiamo se ci sono valori mancanti
missing_train = train_df.isnull().sum().sum()
missing_test = test_df.isnull().sum().sum()

print(f"\n🔍 Controllo qualità dati:")
print(f"   Valori mancanti (train): {missing_train}")
print(f"   Valori mancanti (test): {missing_test}")
print(f"   {'✅ Nessun valore mancante!' if missing_train == 0 and missing_test == 0 else '⚠️  Attenzione: ci sono valori mancanti!'}")

In [ ]:
# Creiamo i DataLoader
batch_size = 64

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,  # Mescoliamo i dati ad ogni epoca
    num_workers=0  # Numero di processi per caricare i dati (0 = sincrono)
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=0
)

print(f"✅ DataLoader creati!")
print(f"📦 Batch size: {batch_size}")
print(f"🔄 Numero di batch nel training: {len(train_loader)}")
print(f"🔄 Numero di batch nel test: {len(test_loader)}")

In [ ]:
import os
import zipfile

# Creiamo la cartella data se non esiste
os.makedirs('data', exist_ok=True)

# Nome del dataset su Kaggle
dataset_name = 'oddrationale/mnist-in-csv'

print("📥 Download del dataset MNIST da Kaggle...")
print(f"   Dataset: {dataset_name}")

try:
    # Scarichiamo il dataset
    api.dataset_download_files(dataset_name, path='data', unzip=True)
    print("✅ Dataset scaricato e decompresso!")
    
    # Elenchiamo i file scaricati
    files = os.listdir('data')
    print(f"\n📂 File nella cartella data/:")
    for f in files:
        size = os.path.getsize(f'data/{f}') / (1024 * 1024)  # Converti in MB
        print(f"   - {f} ({size:.2f} MB)")
        
except Exception as e:
    print(f"❌ Errore durante il download: {e}")
    print("\n💡 Alternativa: scarica manualmente da:")
    print(f"   https://www.kaggle.com/datasets/{dataset_name}")
    print("   e metti i file CSV nella cartella 'data/'")